In [0]:
import dlt
from pyspark.sql import functions as F

# ---------- BRONZE ----------
@dlt.table(name="bronze_reviews", comment="Raw reviews from landing")
def bronze_reviews():
    return (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("pathGlobFilter", "*order_reviews_dataset.csv")   # ← read ONLY the reviews file
        .load("/Volumes/northstar/bronze/landing/"))

# ---------- SILVER (with quality rules) ----------
@dlt.table(name="silver_reviews", comment="Cleaned, validated reviews")
@dlt.expect_or_drop("has_review_id", "review_id IS NOT NULL")
@dlt.expect("valid_score", "review_score BETWEEN 1 AND 5")
def silver_reviews():
    return (dlt.read_stream("bronze_reviews")
        .withColumn("review_score", F.expr("try_cast(review_score as int)"))
        .withColumn("review_creation_date",   F.to_timestamp("review_creation_date"))
        .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))
        .withColumn("review_comment_message",  F.trim("review_comment_message"))
        .dropDuplicates(["review_id"])
        .select("review_id", "order_id", "review_score",                # ✅ only review cols
                "review_comment_title", "review_comment_message",
                "review_creation_date", "review_answer_timestamp"))

# ---------- GOLD (aggregate) ----------
@dlt.table(name="gold_review_summary", comment="Review score distribution")
def gold_review_summary():
    return (dlt.read("silver_reviews")
        .groupBy("review_score").agg(F.count("*").alias("review_count")))